
# Climate Analysis with GeoDataFrame

This notebook replaces `xarray` with `GeoDataFrame` for all climate data operations. It assumes you have a flattened dataset with columns:

- `time`: datetime
- `lat`: latitude
- `lon`: longitude
- `tasmax` or `pr`: climate variable
- `geometry`: Point(lon, lat)


In [ ]:

import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
import matplotlib.pyplot as plt
import glob

# File paths
data_path = "/home/jovyan/shared_materials/climattr_data_please-dont-copy/flattened/"
shapefile_path = "/home/jovyan/shared_exchange/Heatwave_Togo/westafrica"

# Load region shapefile
sf = gpd.read_file(shapefile_path)


In [ ]:

domain = "AFR-22"      # either EUR-11 or AFR-22
varnm = "tasmax"       # either tasmax or pr

units = {"tasmax": "degC", "pr": "mm/day"}[varnm]

# Assume CSVs or GeoParquet containing flattened climate data
files = glob.glob(data_path + f"{varnm}_{domain}_*.parquet")

# Load and combine all files
gdf = pd.concat([gpd.read_parquet(f) for f in files], ignore_index=True)
gdf = gpd.GeoDataFrame(gdf, geometry=gpd.points_from_xy(gdf.lon, gdf.lat), crs="EPSG:4326")

# Ensure time is datetime
gdf["time"] = pd.to_datetime(gdf["time"])


In [ ]:

# Expand bounding box around shapefile by 5 degrees
bounds = sf.geometry.total_bounds  # [minx, miny, maxx, maxy]
buffer = 5
xn, xx, yn, yx = bounds[0] - buffer, bounds[2] + buffer, bounds[1] - buffer, bounds[3] + buffer

# Filter spatially using bounding box
gdf_subset = gdf.cx[xn:xx, yn:yx]


In [ ]:

# Optional: keep only points inside shapefile region
gdf_clipped = gpd.sjoin(gdf_subset, sf, how="inner", predicate="intersects")


In [ ]:

avg_ts = gdf_clipped.groupby("time")[varnm].mean()

plt.figure(figsize=(10, 4))
avg_ts.plot()
plt.title(f"Average {varnm} over region")
plt.ylabel(units)
plt.xlabel("Time")
plt.grid()
plt.show()


In [ ]:

# Example: average over a specific year
selected_year = 2020
gdf_year = gdf_clipped[gdf_clipped["time"].dt.year == selected_year]
gdf_map = gdf_year.groupby(["lat", "lon"])[varnm].mean().reset_index()
gdf_map["geometry"] = gpd.points_from_xy(gdf_map.lon, gdf_map.lat)
gdf_map = gpd.GeoDataFrame(gdf_map, geometry="geometry", crs="EPSG:4326")

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
gdf_map.plot(column=varnm, ax=ax, legend=True, cmap="inferno", markersize=20)
sf.boundary.plot(ax=ax, color="blue", linewidth=1)
plt.title(f"Mean {varnm} for {selected_year}")
plt.show()



This GeoDataFrame-based workflow maintains the key analytical capabilities from the xarray version:
- Spatial filtering using bounding boxes or region masks
- Time-series aggregation over regions
- Spatial mapping for selected periods

If you're working with daily/hourly data, you can extend this to detect heatwaves, thresholds, etc.
